In [ ]:
# Tạo adversarial test set A, B, C, D cho SheepDog từ test.csv
import pandas as pd
import openai
import pickle
from tqdm import tqdm
import concurrent.futures

# Thay bằng API key của bạn
client = openai.OpenAI(api_key='YOUR_API_KEY')

df = pd.read_csv('test.csv')
print('Số mẫu:', len(df))

# Định nghĩa publisher cho từng bộ test
publishers_real = {
    'A': 'National Enquirer',
    'B': 'National Enquirer',
    'C': 'The Sun',
    'D': 'The Sun',
}
publishers_fake = {
    'A': 'CNN',
    'B': 'The New York Times',
    'C': 'CNN',
    'D': 'The New York Times',
}

def generate_rewrite(article, publisher):
    prompt = f"""
    Rewrite the following Vietnamese article in the style of {publisher} (ONLY WRITE IN VIETNAMESE LANGUAGE):
    \n{article}
    """
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=1024,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print('Error:', e)
        return ""

for set_name in ['A']:
    print(f'Generating adversarial test set {set_name}...')
    rewritten = []
    # Tách real/fake
    for label, publisher_map in [(0, publishers_real), (1, publishers_fake)]:
        sub_df = df[df['labels'] == label]
        articles = sub_df['news'].tolist()
        publisher = publisher_map[set_name]
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            for out in tqdm(executor.map(lambda art: generate_rewrite(art, publisher), articles), total=len(articles)):
                rewritten.append(out)
    # Lưu ra file pkl
    with open(f'adversarial_test_{set_name}.pkl', 'wb') as f:
        pickle.dump({'rewritten': rewritten}, f)
    print(f'Saved adversarial_test_{set_name}.pkl')

Số mẫu: 486
Generating adversarial test set A...


100%|██████████| 82/82 [01:40<00:00,  1.22s/it]

Saved adversarial_test_A.pkl
